# TN0 — đối chiếu pipeline đồ án với pipeline MobiVital

Chạy một mạch trên Colab, khoảng 90 phút. Không cần notebook nào chạy trước.

## Ba kiểm tra

- **TN0a** — Tệp lựa chọn kênh do tác giả cung cấp có tái hiện điểm công bố (~0.819) không?
- **TN0b** — Dùng cùng tệp trọng số `.pth`, pipeline đồ án có chọn đúng 537/537 kênh giống pipeline MobiVital không?
- **TN0c** — Train lại LSTM từ đầu thì đạt mức nào? Không bắt buộc giống tuyệt đối tệp trọng số phát hành.

## Hai pipeline

| | code | các tệp chính |
|---|---|---|
| **pipeline MobiVital** | tác giả cung cấp | `inference/evaluate.py`, `inference/mobivital_gen.py`, `training/autoreg_training.py` |
| **pipeline đồ án** | trong `src/` | `scoring.py`, `training.py`, `results.py` |

Pipeline đồ án tồn tại vì code MobiVital chỉ chạy LSTM — `mobivital_gen.py` dòng 152
ghi cứng `LSTMMultiStep(...)`, không hỗ trợ TCN.


## 1. Chuẩn bị Colab, Drive và mã nguồn


In [ ]:
import os
import subprocess
from glob import glob


def sh(cmd):
    """Chạy lệnh, in output thẳng ra màn hình, DỪNG notebook nếu lệnh lỗi."""
    if subprocess.run(cmd, shell=True).returncode != 0:
        raise RuntimeError("Lệnh lỗi: " + cmd)


def grab(cmd, stdout_only=False):
    """Chạy lệnh, TRẢ VỀ output. Dùng cho lệnh ngắn cần lấy kết quả."""
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout, r.stderr)
        raise RuntimeError("Lệnh lỗi: " + cmd)
    return (r.stdout if stdout_only else r.stdout + r.stderr).strip()


REPO = "/content/UWB_RADAR"
if os.path.exists(REPO + "/.git"):
    os.chdir(REPO)
    sh("git pull -q origin main")
else:
    os.chdir("/content")
    sh("rm -rf " + REPO)
    sh("git clone -q https://github.com/quangminhho004-blip/UWB_RADAR.git " + REPO)
    sh("git clone -q https://github.com/nesl/mobivital-public.git " + REPO + "/external/mobivital")
    sh("pip install -q einops")
os.chdir(REPO)

from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/mobivital"
os.makedirs(DRIVE, exist_ok=True)

print("commit đồ án     :", grab("git rev-parse --short HEAD"))
print("commit MobiVital :", grab("git -C external/mobivital rev-parse --short HEAD"))
print("GPU              :", grab("nvidia-smi --query-gpu=name --format=csv,noheader") or "không có")


## 2. Chuẩn bị và kiểm tra dữ liệu chung

Một bản CSV duy nhất trong `external/mobivital/dataset/mobivital/tripod/`. Hai pipeline đọc chung bản đó:

- `prep_breath_final.py` của tác giả → `external/mobivital/data_final/*.npy`
- `scripts/make_npz.py` của đồ án     → `data/processed/by_user/*.npz`

`scripts/check_data.py` so hai bên, phải khớp **từng byte**: ABCDEFKL 1289/1289, GHIJ 537/537.
Mỗi ô tự kiểm tra đủ tệp; thiếu thì chạy lại đúng bước đó, không tin vào dấu vết dở dang.


In [ ]:
# 2.1 — tải và giải nén CSV nếu chưa đủ 1874 tệp
CSV_DIR = "external/mobivital/dataset/mobivital/tripod"

if len(glob(CSV_DIR + "/*.csv")) != 1874:
    sh("apt-get install -qq -y aria2")
    sh("aria2c -x16 -s16 -k5M --summary-interval=0 --console-log-level=warn "
       "-d /content -o tripod.zip "
       "https://zenodo.org/api/records/15022885/files/tripod.zip/content")
    sh("mkdir -p external/mobivital/dataset/mobivital")
    sh("unzip -q -o /content/tripod.zip -d external/mobivital/dataset/mobivital/")

n = len(glob(CSV_DIR + "/*.csv"))
assert n == 1874, "chỉ thấy %d CSV, cần 1874" % n
print(n, "tệp CSV")


In [ ]:
# 2.2 — vá 52 tên tệp lỗi thời, rồi chạy prep_breath_final.py của tác giả
sh("python scripts/mobivital/setup_dataset.py")

NPY = "external/mobivital/data_final"
npy_files = [NPY + "/training_breath_tripod_data.npy",
             NPY + "/testing_breath_tripod_data.npy"]

if not all(os.path.exists(p) for p in npy_files):
    sh("cd external/mobivital && python dataset_preparation/prep_breath_final.py")

for p in npy_files:
    assert os.path.exists(p), "thiếu " + p
print(grab("ls -la " + NPY))


In [ ]:
# 2.3 — pipeline đồ án đọc cùng bộ CSV, rồi so từng byte với data_final
BY_USER = "data/processed/by_user"
want = [c + ".npz" for c in "ABCDEFGHIJKL"]
have = sorted(os.listdir(BY_USER)) if os.path.isdir(BY_USER) else []

if have != want:
    sh("python scripts/make_npz.py")

assert sorted(os.listdir(BY_USER)) == want, "thiếu tệp by_user"
sh("python scripts/check_data.py")   # tự raise nếu không khớp từng byte


In [ ]:
# 2.4 — cắt cửa sổ train cho pipeline đồ án
WIN = "data/processed/windows"
final_train = WIN + "/final_train/train_corr0.9_h200_f25.npz"

if not os.path.exists(final_train) or len(glob(WIN + "/dev_cv/*.npz")) != 8:
    sh("python scripts/make_windows.py")

assert os.path.exists(final_train)
assert len(glob(WIN + "/dev_cv/*.npz")) == 8
print(grab("du -sh %s/dev_cv %s/final_train" % (WIN, WIN)))


## 3. Chạy pipeline MobiVital

Đúng lệnh trong README của tác giả, chạy từ trong thư mục repo của họ, không sửa dòng code nào.

`mobivital_gen.py` luôn ghi tệp lựa chọn kênh ra đúng một tên
`inference/methods/tripod_mobivital_pre_invert_0.9.txt` — tệp này **có sẵn trong repo tác giả**.
Sau mỗi lần chạy: chép kết quả ra `results/` rồi `git checkout` khôi phục tệp gốc,
để repo tác giả không bị sửa (kiểm ở ô 3.4).


In [ ]:
# 3.1 — TN0a: tính điểm từ tệp lựa chọn kênh tác giả cung cấp
os.chdir(REPO)
sh("cp external/mobivital/inference/methods/tripod_mobivital_pre_invert_0.9.txt results/TN0a.txt")

tn0a_mob = float(grab(
    "cd external/mobivital && python -m inference.evaluate "
    "-m tripod_mobivital_pre_invert_0.9.txt -d ./dataset/mobivital/tripod_old_names",
    stdout_only=True).splitlines()[-1])
sh("cp external/mobivital/inference/methods/scores.csv results/scores_TN0a.csv")
print("TN0a  pipeline MobiVital: %.10f" % tn0a_mob)


In [ ]:
# 3.2 — TN0b: chọn kênh từ tệp trọng số tác giả phát hành
os.chdir(REPO + "/external/mobivital")
GEN = "inference/methods/tripod_mobivital_pre_invert_0.9.txt"

sh("python -m inference.mobivital_gen")
sh("cp %s inference/methods/TN0b.txt" % GEN)
sh("cp %s ../../results/TN0b.txt" % GEN)
sh("git checkout -- " + GEN)                       # khôi phục tệp gốc của tác giả

tn0b_mob = float(grab("python -m inference.evaluate -m TN0b.txt --save_file scores_TN0b.csv",
                      stdout_only=True).splitlines()[-1])
sh("cp inference/methods/scores_TN0b.csv ../../results/")
sh("rm inference/methods/TN0b.txt")
print("TN0b  pipeline MobiVital: %.10f" % tn0b_mob)


In [ ]:
# 3.3 — TN0c: train lại LSTM từ đầu (cấu hình optimal_params.json), ~20 phút GPU
os.chdir(REPO + "/external/mobivital")
GEN = "inference/methods/tripod_mobivital_pre_invert_0.9.txt"

sh("python -m training.autoreg_training --model_name lstm_retrained")
sh("python -m inference.mobivital_gen --model_name lstm_retrained")
sh("cp %s inference/methods/TN0c.txt" % GEN)
sh("cp %s ../../results/TN0c.txt" % GEN)
sh("git checkout -- " + GEN)

tn0c_mob = float(grab("python -m inference.evaluate -m TN0c.txt --save_file scores_TN0c.csv",
                      stdout_only=True).splitlines()[-1])
sh("cp inference/methods/scores_TN0c.csv ../../results/")
sh("rm inference/methods/TN0c.txt")
print("TN0c  pipeline MobiVital: %.10f" % tn0c_mob)


In [ ]:
# 3.4 — repo tác giả phải sạch tuyệt đối
os.chdir(REPO)
dirty = grab("git -C external/mobivital status --porcelain")
assert dirty == "", "repo MobiVital bị sửa:\n" + dirty
print("repo MobiVital: SẠCH, không sửa dòng nào")
print("TN0a %.6f   TN0b %.6f   TN0c %.6f   (pipeline MobiVital)"
      % (tn0a_mob, tn0b_mob, tn0c_mob))


## 4. Chạy pipeline đồ án

Ba kiểm tra như trên, bằng code trong `src/`:

| kiểm tra | pipeline MobiVital | pipeline đồ án |
|---|---|---|
| tính điểm từ tệp lựa chọn kênh | `inference/evaluate.py` | `scoring.score_from_txt` |
| chọn kênh từ tệp trọng số | `inference/mobivital_gen.py` | `scoring.score_all` |
| train lại LSTM | `training/autoreg_training.py` | `training.train` |


In [ ]:
os.chdir(REPO)
import csv
import numpy as np
import torch

from src import mobivital_reference as mv
from src import results, scoring, training

TEST_USERS = ["G", "H", "I", "J"]
device = "cuda" if torch.cuda.is_available() else "cpu"


def load_lstm(path):
    model = mv.new_lstm()
    model.load_state_dict(torch.load(path, map_location=device))
    return model.to(device).eval()


def report(name, rows):
    results.save_sessions("results/scores_" + name + ".csv", rows)
    score = float(np.mean([r["pearson"] for r in rows]))
    print("%s  điểm trung bình %d buổi ghi: %.10f" % (name, len(rows), score))
    return score


print("thiết bị:", device)


In [ ]:
# 4.1 — TN0a bằng pipeline đồ án
tn0a_proj = report("ours_a", scoring.score_from_txt("results/TN0a.txt", TEST_USERS))


In [ ]:
# 4.2 — TN0b bằng pipeline đồ án (cùng tệp trọng số tác giả phát hành), ~10 phút
rows_b = scoring.score_all(
    TEST_USERS, load_lstm("external/mobivital/checkpoints/lstm_pred_tripod_0.9.pth"))
scoring.write_txt(rows_b, "results/ours_b.txt")
tn0b_proj = report("ours_b", rows_b)


In [ ]:
# 4.3 — TN0c bằng pipeline đồ án (train lại), ~25 phút GPU
X, y = training.load_windows(["train"], folder="data/processed/windows/final_train")
print(X.shape[0], "cửa sổ train")

training.set_seed(1234)
train_result = training.train(mv.new_lstm(), training.make_loader(X, y), None,
                              "runs/tn0/ours_c", loss_name="mse")
results.save_curve("runs/tn0/ours_c/curve.csv", train_result["curve"])

rows_c = scoring.score_all(TEST_USERS, load_lstm(train_result["final_path"]))
scoring.write_txt(rows_c, "results/ours_c.txt")
tn0c_proj = report("ours_c", rows_c)


## 5. So sánh hai pipeline

- **TN0a** phải khớp: cùng tệp lựa chọn kênh, cùng cách tính điểm.
- **TN0b** phải khớp 537/537 kênh và chênh lệch điểm gần bằng 0: cùng tệp trọng số, không có gì ngẫu nhiên.
- **TN0c** chỉ tham khảo: vòng train hai pipeline khác nhau ở thứ tự xáo trộn dữ liệu.


In [ ]:
def scores_mob(p):    # cột 0 tên tệp CSV, cột 1 điểm
    return {r[0]: float(r[1]) for r in list(csv.reader(open("results/" + p)))[1:]}

def scores_proj(p):
    return {r["session_file"]: float(r["pearson"]) for r in csv.DictReader(open("results/" + p))}

def picks(p):         # tên tệp CSV -> (kênh khoảng cách, phép biến đổi)
    return {r[0]: (r[1], r[2]) for r in csv.reader(open("results/" + p))}


pm, pp = picks("TN0b.txt"), picks("ours_b.txt")
same_b = sum(1 for f in pm if pm[f] == pp.get(f))

sm, sp = scores_mob("scores_TN0b.csv"), scores_proj("scores_ours_b.csv")
gap_b = max(abs(sm[f] - sp[f]) for f in sm)
gap_a = abs(tn0a_mob - tn0a_proj)
git_clean = grab("git -C external/mobivital status --porcelain") == ""

ok_a = gap_a < 1e-9
ok_b = (same_b == len(pm)) and (gap_b < 1e-9)


def verdict(ok):
    return "ĐẠT" if ok else "KHÔNG ĐẠT"


print("%-26s %-13s %-15s %s" % ("Kiểm tra", "MobiVital", "Pipeline đồ án", "Kết luận"))
print("-" * 72)
print("%-26s %-13.6f %-15.6f %s" % ("Điểm từ TXT tác giả", tn0a_mob, tn0a_proj, verdict(ok_a)))
print("%-26s %-13.6f %-15.6f %s" % ("Cùng tệp trọng số", tn0b_mob, tn0b_proj,
                                    "%d/%d kênh — %s" % (same_b, len(pm), verdict(ok_b))))
print("%-26s %-13.6f %-15.6f %s" % ("Train lại LSTM", tn0c_mob, tn0c_proj, "thông tin tham khảo"))
print("-" * 72)
print("repo MobiVital không bị sửa : %s" % verdict(git_clean))
print("chênh lệch điểm TN0b lớn nhất: %.2e" % gap_b)

if not (ok_a and ok_b and git_clean):
    raise RuntimeError("TN0 KHÔNG ĐẠT — xem bảng trên")
print("\nTN0 ĐẠT")


## 6. Lưu toàn bộ kết quả


In [ ]:
os.chdir(REPO)
sh("tar -czf %s/tn0.tar.gz results runs/tn0" % DRIVE)
print(grab("ls -la %s/tn0.tar.gz" % DRIVE))
print()
print(grab("ls results"))
